In [6]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving player_injuries_impact.csv to player_injuries_impact.csv


In [8]:
df = pd.read_csv('player_injuries_impact.csv')
df.head()

,Name,Team Name,Position,Age,Season,FIFA rating,Injury,Date of Injury,Date of return,Match1_before_injury_Result,...,Match1_after_injury_GD,Match1_after_injury_Player_rating,Match2_after_injury_Result,Match2_after_injury_Opposition,Match2_after_injury_GD,Match2_after_injury_Player_rating,Match3_after_injury_Result,Match3_after_injury_Opposition,Match3_after_injury_GD,Match3_after_injury_Player_rating
0,Jamaal Lascelles,Newcastle,Center Back,26,2019/20,77,Knee injury,"Nov 9, 2019","Jan 13, 2020",draw,...,1,7.1,draw,Everton,0,6.2,draw,Norwich City,0,6.7
1,Fabian Schär,Newcastle,Center Back,28,2019/20,79,Knee injury,"Oct 20, 2019","Nov 24, 2019",lose,...,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.
2,Fabian Schär,Newcastle,Center Back,28,2019/20,79,Hamstring strain,"Jan 2, 2020","Jan 17, 2020",lose,...,0,6(S),lose,Arsenal,-4,N.A.,lose,Crystal Palace,-1,6.5
3,Fabian Schär,Newcastle,Center Back,28,2019/20,79,Shoulder injury,"Jul 16, 2020","Sep 28, 2020",lose,...,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.
4,Paul Dummett,Newcastle,Center Back,28,2019/20,75,Groin injury,"Dec 22, 2019","Jan 10, 2020",win,...,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 656 entries, 0 to 655
Data columns (total 42 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Name                                656 non-null    object
 1   Team Name                           656 non-null    object
 2   Position                            656 non-null    object
 3   Age                                 656 non-null    int64 
 4   Season                              656 non-null    object
 5   FIFA rating                         656 non-null    int64 
 6   Injury                              656 non-null    object
 7   Date of Injury                      656 non-null    object
 8   Date of return                      656 non-null    object
 9   Match1_before_injury_Result         656 non-null    object
 10  Match1_before_injury_Opposition     656 non-null    object
 11  Match1_before_injury_GD             656 non-null    object

In [11]:
df.describe()

,Age,FIFA rating
count,656.000000,656.000000
mean,26.661585,78.576220
std,3.580028,4.108117
min,18.000000,66.000000
25%,24.000000,76.000000
50%,27.000000,79.000000
75%,29.000000,81.000000
max,39.000000,90.000000


In [12]:
df.isnull().sum()

,0
Name,0
Team Name,0
Position,0
Age,0
Season,0
FIFA rating,0
Injury,0
Date of Injury,0
Date of return,0
Match1_before_injury_Result,0


In [13]:
df.replace("NA", np.nan, inplace=True)
df.replace("N.A.", np.nan, inplace=True)
df.replace("na", np.nan, inplace=True)

In [14]:
df.isnull().sum()

,0
Name,0
Team Name,0
Position,0
Age,0
Season,0
FIFA rating,0
Injury,0
Date of Injury,0
Date of return,0
Match1_before_injury_Result,65


In [16]:
before_cols = [col for col in df.columns if 'before' in col.lower() and 'rating' in col.lower()]
after_cols = [col for col in df.columns if 'after' in col.lower() and 'rating' in col.lower()]

In [17]:
print(before_cols)
print(after_cols)

['Match1_before_injury_Player_rating', 'Match2_before_injury_Player_rating', 'Match3_before_injury_Player_rating']
['Match1_after_injury_Player_rating', 'Match2_after_injury_Player_rating', 'Match3_after_injury_Player_rating']


In [19]:
for col in before_cols + after_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [20]:
df[before_cols].dtypes

,0
Match1_before_injury_Player_rating,float64
Match2_before_injury_Player_rating,float64
Match3_before_injury_Player_rating,float64


In [21]:
df['avg_before'] = df[before_cols].mean(axis=1)
df['avg_after'] = df[after_cols].mean(axis=1)

In [22]:
df[['avg_before', 'avg_after']].head()

,avg_before,avg_after
0,6.433333,6.666667
1,6.466667,NaN
2,6.066667,6.500000
3,5.933333,NaN
4,6.000000,NaN


In [23]:
df = df.dropna(subset=['avg_after'])

In [24]:
df[['avg_before', 'avg_after']].head()

,avg_before,avg_after
0,6.433333,6.666667
2,6.066667,6.500000
5,6.266667,6.566667
7,6.500000,6.800000
8,6.000000,6.200000


In [25]:
df['performance_drop'] = df['avg_before'] - df['avg_after']

In [27]:
df[['avg_before', 'avg_after', 'performance_drop']].head()

,avg_before,avg_after,performance_drop
0,6.433333,6.666667,-0.233333
2,6.066667,6.500000,-0.433333
5,6.266667,6.566667,-0.300000
7,6.500000,6.800000,-0.300000
8,6.000000,6.200000,-0.200000


In [28]:
def performance_label(x):
    if x > 0:
        return "Dropped"
    elif x < 0:
        return "Improved"
    else:
        return "No Change"

df['performance_change'] = df['performance_drop'].apply(performance_label)

In [29]:
df[['performance_drop', 'performance_change']].head()

,performance_drop,performance_change
0,-0.233333,Improved
2,-0.433333,Improved
5,-0.300000,Improved
7,-0.300000,Improved
8,-0.200000,Improved


In [30]:
missed_result_cols = [col for col in df.columns if 'missed_match' in col.lower() and 'result' in col.lower()]
print(missed_result_cols)

['Match1_missed_match_Result', 'Match2_missed_match_Result', 'Match3_missed_match_Result']


In [31]:
missed_df = df[missed_result_cols].copy()

missed_long = missed_df.melt(value_name='Result')

In [32]:
missed_long = missed_long.dropna()

In [33]:
missed_long['Result'].value_counts()

,count
Result,
win,409
lose,394
draw,267


In [34]:
result_percent = missed_long['Result'].value_counts(normalize=True) * 100
print(result_percent)

Result
win     38.224299
lose    36.822430
draw    24.953271
Name: proportion, dtype: float64


In [35]:
df['Date of Injury'] = pd.to_datetime(df['Date of Injury'], errors='coerce')
df['Date of return'] = pd.to_datetime(df['Date of return'], errors='coerce')

df['recovery_days'] = (df['Date of return'] - df['Date of Injury']).dt.days

In [37]:
final_cols = [
    'Name',
    'Team Name',
    'Age',
    'Position',
    'Injury',
    'avg_before',
    'avg_after',
    'performance_drop',
    'performance_change',
    'recovery_days'
]

df_final = df[final_cols]

In [38]:
df_final.head()

,Name,Team Name,Age,Position,Injury,avg_before,avg_after,performance_drop,performance_change,recovery_days
0,Jamaal Lascelles,Newcastle,26,Center Back,Knee injury,6.433333,6.666667,-0.233333,Improved,65.0
2,Fabian Schär,Newcastle,28,Center Back,Hamstring strain,6.066667,6.500000,-0.433333,Improved,15.0
5,Ciaran Clark,Newcastle,30,Center Back,Calf injury,6.266667,6.566667,-0.300000,Improved,32.0
7,Jetro Willems,Newcastle,26,Left Back,Knee injury,6.500000,6.800000,-0.300000,Improved,13.0
8,Jetro Willems,Newcastle,26,Left Back,Knee injury,6.000000,6.200000,-0.200000,Improved,10.0


In [40]:
df['injury_month'] = df['Date of Injury'].dt.month
df['injury_year'] = df['Date of Injury'].dt.year

In [41]:
df['injury_month_name'] = df['Date of Injury'].dt.strftime('%B')

In [42]:
def age_group(age):
    if age < 23:
        return "Young"
    elif age < 30:
        return "Prime"
    else:
        return "Senior"

df['age_group'] = df['Age'].apply(age_group)

In [43]:
df['injury_count'] = 1

In [44]:
final_cols = [
    'Name',
    'Team Name',
    'Age',
    'age_group',
    'Position',
    'Injury',
    'injury_month_name',
    'injury_year',
    'avg_before',
    'avg_after',
    'performance_drop',
    'performance_change',
    'recovery_days',
    'injury_count'
]

df_final = df[final_cols]

In [45]:
df_final.isnull().sum()

,0
Name,0
Team Name,0
Age,0
age_group,0
Position,0
Injury,0
injury_month_name,77
injury_year,77
avg_before,51
avg_after,0


In [46]:
df_final.dtypes

,0
Name,object
Team Name,object
Age,int64
age_group,object
Position,object
Injury,object
injury_month_name,object
injury_year,float64
avg_before,float64
avg_after,float64


In [47]:
df_final['Injury'] = df_final['Injury'].str.title()
df_final['Position'] = df_final['Position'].str.title()
df_final['Team Name'] = df_final['Team Name'].str.title()

/tmp/ipykernel_5652/264154182.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Injury'] = df_final['Injury'].str.title()
/tmp/ipykernel_5652/264154182.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Position'] = df_final['Position'].str.title()
/tmp/ipykernel_5652/264154182.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.py

In [48]:
df['injury_month_num'] = df['Date of Injury'].dt.month
df_final['injury_month_num'] = df['injury_month_num']

/tmp/ipykernel_5652/3888529473.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['injury_month_num'] = df['injury_month_num']


In [49]:
df_final.duplicated().sum()

np.int64(0)

In [50]:
df_final.head()

,Name,Team Name,Age,age_group,Position,Injury,injury_month_name,injury_year,avg_before,avg_after,performance_drop,performance_change,recovery_days,injury_count,injury_month_num
0,Jamaal Lascelles,Newcastle,26,Prime,Center Back,Knee Injury,November,2019.0,6.433333,6.666667,-0.233333,Improved,65.0,1,11.0
2,Fabian Schär,Newcastle,28,Prime,Center Back,Hamstring Strain,January,2020.0,6.066667,6.500000,-0.433333,Improved,15.0,1,1.0
5,Ciaran Clark,Newcastle,30,Senior,Center Back,Calf Injury,December,2019.0,6.266667,6.566667,-0.300000,Improved,32.0,1,12.0
7,Jetro Willems,Newcastle,26,Prime,Left Back,Knee Injury,September,2019.0,6.500000,6.800000,-0.300000,Improved,13.0,1,9.0
8,Jetro Willems,Newcastle,26,Prime,Left Back,Knee Injury,December,2019.0,6.000000,6.200000,-0.200000,Improved,10.0,1,12.0


In [51]:
df_final['injury_year'] = df_final['injury_year'].astype('Int64')

/tmp/ipykernel_5652/528869499.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['injury_year'] = df_final['injury_year'].astype('Int64')


In [52]:
df_final['avg_before'] = df_final['avg_before'].round(2)
df_final['avg_after'] = df_final['avg_after'].round(2)
df_final['performance_drop'] = df_final['performance_drop'].round(2)
df_final['recovery_days'] = df_final['recovery_days'].round(0)

/tmp/ipykernel_5652/2746806898.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['avg_before'] = df_final['avg_before'].round(2)
/tmp/ipykernel_5652/2746806898.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['avg_after'] = df_final['avg_after'].round(2)
/tmp/ipykernel_5652/2746806898.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

In [53]:
df_final.to_csv("final_injury_analysis.csv", index=False)

In [54]:
from google.colab import files
files.download("final_injury_analysis.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>